# LatentMAS Instrumented Run — Kaggle

Clones the repo, runs `instrumented_run.py` across all three tasks sequentially on a single GPU, and saves all activations needed for the offline interpretability experiments.

**Output layout** (`/kaggle/working/activations/`):
```
activations/
  wa_matrix.pt              ← W_a alignment matrix (once)
  metadata.json             ← run config
  gsm8k/
    metadata.csv
    example_0000/
      latent_thoughts.pt    ← [A, m, D] pre/post W_a
      latent_per_layer.pt   ← [A, m, L+1, D] all layers
      prompt_hidden.pt      ← last 64 prompt token hidden states
      kv_latent.pt          ← KV cache at latent positions
      text_outputs.json     ← decoded text + top-5 vocab tokens
      metadata.json         ← correct / gold / pred
  arc_challenge/
  mbppplus/
```

**Resumable**: re-run any cell after a timeout — existing `example_XXXX/` folders are skipped.

**Download**: after the run, use the zip cell at the bottom then grab `activations.zip` from the **Output** tab.

## 1 — Install dependencies

In [ ]:
!pip install -q transformers accelerate datasets tqdm huggingface_hub

## 2 — Clone repo

In [ ]:
import os, sys

REPO_URL  = "https://github.com/spraldev/LatentMAS-interp"
REPO_PATH = "/kaggle/working/repo"

if not os.path.isdir(REPO_PATH):
    !git clone {REPO_URL} {REPO_PATH}
else:
    print("Repo already cloned — pulling latest")
    !git -C {REPO_PATH} pull

sys.path.insert(0, REPO_PATH)
os.chdir(REPO_PATH)
print("Files:", os.listdir("."))

## 3 — HuggingFace login

Add your HF read token in the Kaggle **Secrets** panel (the key icon on the left sidebar) with the name `HF_TOKEN`.

In [ ]:
from huggingface_hub import login
import os

# Kaggle Secrets → Add secret named HF_TOKEN
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Logged in to HuggingFace.")
else:
    print("WARNING: No HF_TOKEN found. Run may fail if model is gated.")

## 4 — Config

In [ ]:
import torch

OUTPUT_DIR   = "/kaggle/working/activations"
MODEL_NAME   = "Qwen/Qwen3-4B"
MAX_SAMPLES  = 500      # per task
LATENT_STEPS = 4
MAX_TOKENS   = 2048
SEED         = 42

print(f"GPUs available: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  cuda:{i} — {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory // 1024**3} GB)")

## 5 — Test run (5 examples, ~5 min)

Run this first to verify the pipeline on a single GPU before the full parallel run.

In [ ]:
!python {REPO_PATH}/instrumented_run.py \
    --output_dir {OUTPUT_DIR}_test \
    --model_name {MODEL_NAME} \
    --tasks gsm8k \
    --latent_steps {LATENT_STEPS} \
    --max_new_tokens 512 \
    --latent_space_realign \
    --device cuda:0 \
    --seed {SEED} \
    --test

In [ ]:
# Sanity check — print shapes for example_0000
import torch, json
from pathlib import Path

test_root = Path(f"{OUTPUT_DIR}_test")
print("=== Output dir contents ===")
if not test_root.exists():
    print("ERROR: output dir does not exist — the test run above failed.")
    print("Scroll up and check cell 5 output for the actual error message.")
    raise SystemExit

for p in sorted(test_root.rglob("*"))[:40]:
    print(" ", p.relative_to(test_root))

test_ex = test_root / "gsm8k" / "example_0000"
if not test_ex.exists():
    print(f"\nERROR: {test_ex} not found — check cell 5 output above for the error.")
    raise SystemExit

print("\n=== example_0000 shapes ===")
lt = torch.load(test_ex / "latent_thoughts.pt", map_location="cpu", weights_only=False)
print("pre_aligned :", lt["pre_aligned"].shape,  " → [agents, m, d_h]")
print("post_aligned:", lt["post_aligned"].shape, " → [agents, m, d_h]")
print("agents      :", lt["agents"])

with open(test_ex / "metadata.json") as f:
    print(json.dumps(json.load(f), indent=2))

## 6 — Full run (all 3 tasks, single GPU)

Runs all three tasks sequentially on `cuda:0`:
- `gsm8k` → `arc_challenge` → `mbppplus`

500 examples per task, ~25s each → **~10–12 hours total**.

Log is written to `run.log` in the output dir.

In [ ]:
import subprocess, time, os
from pathlib import Path

os.makedirs(OUTPUT_DIR, exist_ok=True)

args = [
    "python", f"{REPO_PATH}/instrumented_run.py",
    "--output_dir",     OUTPUT_DIR,
    "--model_name",     MODEL_NAME,
    "--tasks",          "gsm8k", "arc_challenge", "mbppplus",
    "--max_samples",    str(MAX_SAMPLES),
    "--latent_steps",   str(LATENT_STEPS),
    "--max_new_tokens", str(MAX_TOKENS),
    "--latent_space_realign",
    "--device",         "cuda:0",
    "--seed",           str(SEED),
    "--log_file",       f"{OUTPUT_DIR}/run.log",
    "--storage_warn_gb", "17.0",
    "--check_storage_every", "25",
    "--verify_every",   "10",
]

proc = subprocess.Popen(args, env=os.environ.copy())
print(f"Launched (pid={proc.pid})")

start = time.time()
while proc.poll() is None:
    time.sleep(60)
    elapsed = (time.time() - start) / 3600
    print(f"[{elapsed:.1f}h] still running...")

status = "OK" if proc.returncode == 0 else f"FAILED (exit {proc.returncode})"
print(f"\nDone — {status}")

## 7 — Storage + accuracy report

In [ ]:
import csv
from pathlib import Path

root = Path(OUTPUT_DIR)

def du_gb(path):
    return sum(p.stat().st_size for p in Path(path).rglob("*") if p.is_file()) / 1024**3

print(f"Total disk usage: {du_gb(root):.2f} GB")
print()
for task_dir in sorted(d for d in root.iterdir() if d.is_dir()):
    n_examples = len([d for d in task_dir.iterdir() if d.is_dir()])
    size = du_gb(task_dir)
    csv_path = task_dir / "metadata.csv"
    if csv_path.exists():
        rows = list(csv.DictReader(open(csv_path)))
        correct = sum(1 for r in rows if r.get("correct", "").lower() == "true")
        n = len(rows)
        acc = f"{correct}/{n} ({100*correct/n:.1f}%)"
    else:
        acc = "no metadata.csv yet"
    print(f"  {task_dir.name:15s}  {size:.2f} GB   {n_examples} examples   acc={acc}")

## 8 — Zip and download

After zipping, go to your notebook page → **Output** tab → `activations.zip` → download icon.

Or via Kaggle CLI on your local machine:
```bash
pip install kaggle
kaggle kernels output YOUR_USERNAME/YOUR_NOTEBOOK_SLUG -p ./downloads/
```

In [ ]:
import shutil, os

print("Zipping activations/ — may take a few minutes...")
shutil.make_archive("/kaggle/working/activations", "zip", "/kaggle/working", "activations")
size_gb = os.path.getsize("/kaggle/working/activations.zip") / 1024**3
print(f"Done — activations.zip  ({size_gb:.2f} GB)")
print("Download: Output tab → activations.zip → download icon")

## 9 — How to load locally (reference)

```python
import torch, json, pandas as pd
from pathlib import Path

ROOT = Path("activations")

# Per-task accuracy table
df = pd.read_csv(ROOT / "gsm8k" / "metadata.csv")
print(df[["example_id", "correct", "prediction", "gold"]].head())

# Load one example
ex = ROOT / "gsm8k" / "example_0000"

lt  = torch.load(ex / "latent_thoughts.pt",  map_location="cpu", weights_only=False)
ll  = torch.load(ex / "latent_per_layer.pt", map_location="cpu", weights_only=False)
ph  = torch.load(ex / "prompt_hidden.pt",    map_location="cpu", weights_only=False)
kv  = torch.load(ex / "kv_latent.pt",        map_location="cpu", weights_only=False)

# lt["pre_aligned"]          [A, m, D]       hidden states before W_a   (Exp 1, 8, 26)
# lt["post_aligned"]         [A, m, D]       injected latent thoughts   (Exp 2, 3, 6, 9, 10, 15-27)
# ll["hidden_per_layer"]     [A, m, L+1, D]  all transformer layers     (Exp 5, 7, 11, 19, 20)
# ph["planner"]              [64, D]         last 64 prompt tokens      (Exp 3, 9, 24)
# kv["planner"]              list[(k,v)]     KV cache at latent pos     (Exp 6, 11, 25)

# W_a matrix
wa = torch.load(ROOT / "wa_matrix.pt", map_location="cpu", weights_only=False)
W_a = wa["W_a"]   # [D, D]
```